# QueryMind ML — Training walkthrough & MLOps

This notebook runs **the same code path as regular execution** (`run_pipeline.py`): for each of the six activities it

1. loads the feature view,
2. trains + evaluates the model,
3. registers a version in `ml_model_registry` (joblib artifact + metrics),
4. writes predictions into the activity's `ml_*` output table (replacing only that activity's window),
5. records distribution/drift stats in `ml_prediction_monitor` and a run row in `ml_pipeline_run`.

**Prerequisites:** `python run_pipeline.py --step sql`, `DATABASE_URL` set, dependencies from `ml_pipeline/requirements.txt` installed.

> ⚠️ This notebook writes to the database (same as the scheduled pipeline). Use a dev Neon branch if you want to keep production outputs untouched.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ML_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ML_ROOT))

from querymind_ml import config, db
from querymind_ml.activities import ACTIVITIES
from run_pipeline import run_activity

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (11, 4)

conn = db.connect(config.get_dsn()).__enter__()

def q(sql: str) -> pd.DataFrame:
    return db.read_frame(conn, sql)

def run(name: str) -> dict:
    summary = run_activity(name, "all", conn)
    print(f"{name}: {summary['headline']} | written={summary['written']} | {summary['model_version']}")
    return summary

print("activities:", ", ".join(ACTIVITIES))

## 1. Tool adoption forecast

`HistGradientBoostingRegressor` on adoption-rate lags + seasonality + static tool attributes, forecast recursively 12 months ahead with residual-based intervals.

In [ ]:
s1 = run("tool_adoption_forecast")
pd.Series(s1["metrics"])

In [ ]:
hist = q("""
SELECT forecast_month, tool_name, label_adoption_rate AS actual
FROM ml_features_tool_adoption
""")
fut = q("""
SELECT f.forecast_month, t.tool_name, f.predicted_adoption_rate,
       f.lower_bound, f.upper_bound, f.model_version
FROM ml_tool_adoption_forecast f
JOIN ai_tools t ON t.tool_id = f.tool_id
""")
name = fut.tool_name.iloc[0]
h = hist[hist.tool_name == name].set_index("forecast_month")["actual"]
f = fut[fut.tool_name == name].set_index("forecast_month")
ax = h.plot(label="actual")
f["predicted_adoption_rate"].plot(ax=ax, style="--", label="forecast")
ax.fill_between(f.index, f["lower_bound"], f["upper_bound"], alpha=0.2)
ax.set_title(f"Adoption forecast — {name} ({f.model_version.iloc[0]})"); ax.legend();

## 2. User adoption risk

`HistGradientBoostingClassifier` (LOW/MEDIUM/HIGH). `risk_score` = probability-weighted expected risk in [0,1]; `risk_band` = argmax class.

In [ ]:
s2 = run("user_adoption_risk")
print("metrics:", json.dumps(s2["metrics"], indent=2))

In [ ]:
written = q("""
SELECT risk_band, COUNT(*) AS users, AVG(risk_score) AS avg_score
FROM ml_user_adoption_risk
WHERE prediction_month >= date_trunc('month', CURRENT_DATE)::date
GROUP BY risk_band ORDER BY users DESC
""")
print(written)
q("""
SELECT d.department_name, ROUND(AVG(r.risk_score), 4) AS avg_risk,
       SUM(CASE WHEN r.risk_band = 'HIGH' THEN 1 ELSE 0 END) AS high_risk_users
FROM ml_user_adoption_risk r
JOIN users u ON u.user_id = r.user_id
JOIN departments d ON d.department_id = u.department_id
WHERE r.prediction_month >= date_trunc('month', CURRENT_DATE)::date
GROUP BY d.department_name ORDER BY avg_risk DESC
""")

## 3. User segmentation

`StandardScaler` + `KMeans`, k chosen by silhouette (canonical k=5 preferred on ties); cluster names assigned by interpretable rules (Power User / Multi-Tool / Specialized / Explorer / Occasional).

In [ ]:
s3 = run("user_segments")
pd.Series(s3["metrics"])

In [ ]:
q("""
SELECT s.segment_name, COUNT(*) AS users,
       ROUND(AVG(s.segment_score), 4) AS avg_score,
       ROUND(AVG(f.total_events), 1) AS avg_events,
       ROUND(AVG(f.distinct_tools), 2) AS avg_tools
FROM ml_user_segments s
JOIN ml_features_user_segments f ON f.user_id = s.user_id
WHERE s.prediction_month >= date_trunc('month', CURRENT_DATE)::date
GROUP BY s.segment_name ORDER BY avg_events DESC
""")

## 4. Usage anomaly detection

`IsolationForest` over the residual space of the weekly baseline (actual, 4-week mean, residual, ratio, z-score). Evaluation compares flagged weeks with injected surge ground truth and with a naive `z ≥ 3` baseline.

In [ ]:
s4 = run("usage_anomalies")
print(json.dumps(s4["metrics"], indent=2))

In [ ]:
q("""
SELECT a.metric_date, tm.team_name, tl.tool_name,
       a.actual_value, a.expected_value, a.anomaly_score
FROM ml_usage_anomalies a
JOIN teams tm ON tm.team_id = a.team_id
JOIN ai_tools tl ON tl.tool_id = a.tool_id
WHERE a.metric_name = 'usage_volume_weekly' AND a.is_anomaly
ORDER BY a.anomaly_score DESC
LIMIT 15
""")

## 5. Team productivity forecast

`HistGradientBoostingRegressor` on throughput lags (1..6 months) + delivery health; recursive 6-month horizon with residual intervals; complete months only.

In [ ]:
s5 = run("team_productivity_forecast")
pd.Series(s5["metrics"])

In [ ]:
team = q("SELECT team_name FROM teams ORDER BY team_id").team_name.iloc[0]
hist = q("""
SELECT metric_month, label_completed_story_points
FROM ml_features_team_productivity ORDER BY metric_month
""")
fut = q("""
SELECT f.prediction_month, f.predicted_completed_story_points, f.lower_bound, f.upper_bound
FROM ml_team_productivity_forecast f JOIN teams t ON t.team_id = f.team_id
WHERE t.team_name = '%s' ORDER BY f.prediction_month
""" % team)
ax = hist.set_index("metric_month")["label_completed_story_points"].plot(label="actual")
fut.set_index("prediction_month")["predicted_completed_story_points"].plot(ax=ax, style="--", label="forecast")
ax.fill_between(fut.prediction_month, fut.lower_bound, fut.upper_bound, alpha=0.2)
ax.set_title(f"Throughput forecast — {team}"); ax.legend();

## 6. Tool recommendations

Item-item collaborative filtering (cosine over success-weighted 90-day interactions) blended with popularity + category fit; candidates are tools below the engagement share. Evaluated on *new-tool* adoption in the last 30 days (precision@5 / recall@5 / MAP@5).

In [ ]:
s6 = run("tool_recommendations")
pd.Series(s6["metrics"])

In [ ]:
q("""
SELECT u.employee_code, d.department_name, t.tool_name,
       r.recommendation_score, r.recommendation_reason
FROM ml_tool_recommendations r
JOIN users u ON u.user_id = r.user_id
JOIN departments d ON d.department_id = u.department_id
JOIN ai_tools t ON t.tool_id = r.tool_id
ORDER BY r.recommendation_score DESC
LIMIT 15
""")

## MLOps: registry, monitoring, regular execution

Everything above is what the scheduled runner does. The operational surface:

| Concern | Where |
|---|---|
| Run audit | `ml_pipeline_run` (activity, step, status, model_version, details) |
| Model registry | `ml_model_registry` + `artifacts/<activity>/<version>/{model.joblib, meta.json}` |
| Champion / challenger | first model auto-promotes; later runs register as `challenger`; promote with `--promote` |
| Prediction drift | `ml_prediction_monitor` (score mean/p50/p95, band mix, shift flags vs previous run) |
| Serving step | `--step predict` loads the champion without retraining |

**Suggested cadence** (cron on any host that can reach Neon):

```cron
# daily 06:00 — refresh predictions from the current champion
0 6 * * *  cd ml_pipeline && python run_pipeline.py --step predict
# Mondays 07:00 — full retrain, register, rewrite outputs, monitor
0 7 * * 1  cd ml_pipeline && python run_pipeline.py --step all
# monthly — extend history before retraining (optional, idempotent)
# (run sql/01 manually or via: python run_pipeline.py --step sql)
```

**Health checks after each run:**

```sql
SELECT activity, model_version, status, trained_at FROM ml_model_registry ORDER BY trained_at DESC;
SELECT activity, monitor_date, score_mean, drift_vs_previous FROM ml_prediction_monitor ORDER BY monitor_id DESC;
SELECT activity, step, status, started_at FROM ml_pipeline_run ORDER BY run_id DESC LIMIT 20;
```

Drift flags (`score_mean_shift_flag`, `band_share_delta`) tell you when to investigate; a drop in holdout metrics in `ml_model_registry.metrics` tells you when to retrain or revisit features.